# Back-of-the-envelope estimation for AI workloads

> 📓 *Companion to* **Modern Agentic AI Engineer** · Ch 42 §42.3 (and the §42.7 worked example) · type: concept-lab

**The promise:** by the end you can size any agentic feature in ten minutes — peak call rate, tokens/day, \$/day and \$/task, vector and trace storage — from a handful of assumptions, and *name the multiplier that dominates the bill* before you write a line of code.

This runs fully offline and free. It is pure arithmetic — no API key, no network, just the standard library. You change the assumptions and re-read until the *numbers*, not your instincts, kill the bad design (§42.1).

## 🧠 Why this matters

Estimation is the step most engineers skip and the one that most reliably kills bad designs early. The arithmetic is deliberately crude — powers of ten, not decimal places — and takes ten minutes, but it answers the questions that decide the whole architecture: can this be a blocking request–response API? Does the unit economics print money or die on arrival? Does the trace store need tiering?

The one mental model to carry: spend scales as **users × requests × loop turns × tokens/turn × price** — four innocent multipliers, each harmless alone, that compound into the bill. Estimate at the *token* level, never the request level, or you will be wrong by the exact factor that matters (§42.3 ⚠️). This notebook turns the chapter's back-of-envelope math into a calculator you can point at your own feature.

## Objectives & prereqs

**By the end you can:**
- Encode a feature as a few assumptions and get traffic, tokens, \$/day, \$/task, and storage out.
- Reproduce the chapter's 10k-DAU scenario (≈ \$4,000/day, ≈ \$0.02/request) and confirm it.
- Show from latency physics alone why an agent loop *cannot* be a blocking request–response API.
- Run a sensitivity sweep and watch a +1,000-token prompt add ≈ \$800/day (§42.3's pitfall, reproduced).
- Name the *binding multiplier* for a design before building anything.

**Prereqs:** Chapter 40 (cost/latency/performance) read; Chapter 30 (data layer) for the storage math. No prior notebook required — this is Part XI's recommended entry point.

**Packages:** standard library only (`dataclasses`, `math`). Nothing beyond `requirements.txt`; no API key; fully offline.

In [ ]:
# Setup — imports, env, and the MOCK switch.
import math
import os
from dataclasses import dataclass

from dotenv import load_dotenv

load_dotenv()  # reads a local .env if present; never hardcode keys

# Every notebook in this repo honors COMPANION_MOCK. This one is pure local
# arithmetic — it makes no model calls in EITHER mode — but we read the switch so the
# habit and the message stay consistent across the whole course. There is nothing to
# pay for and nothing to seed: integer/float math is already deterministic.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

print("Estimation calculator — offline, free, no API key needed.")
print(f"COMPANION_MOCK = {os.getenv('COMPANION_MOCK', '1')} (this notebook makes no model calls either way)")

## 🧠 The anchors: memorize five numbers, and estimation becomes mental arithmetic

The chapter's 💡 tip: a handful of anchors turn every estimate into arithmetic you can do in a meeting. We encode them once as constants — these are the *only* magic numbers in the notebook, and they match §42.3 exactly.

| Anchor | Value | Why it's useful |
|---|---|---|
| Seconds in a day | 86,400 | turns “per day” into “per second” |
| Word ≈ tokens | 1.3 | size prose without a tokenizer |
| Page ≈ tokens | 500 | size documents and corpora |
| 1,536-dim float32 embedding | ≈ 6 KB | size a vector store (1,536 × 4 bytes) |
| Output speed | ≈ 60 tok/s | turn output tokens into wall-clock latency |

Prices move and speeds move; the *method* doesn't. We use the chapter's round illustrative prices: \$1.00 / million input tokens, \$5.00 / million output tokens.

In [ ]:
# The anchors from §42.3 — round, illustrative, and the only constants here.
SECONDS_PER_DAY = 86_400          # a day in seconds
TOKENS_PER_WORD = 1.3             # English prose, rough
TOKENS_PER_PAGE = 500             # a page of text
EMBEDDING_BYTES = 1_536 * 4       # 1,536-dim float32 vector ≈ 6,144 bytes ≈ 6 KB
OUTPUT_TOKENS_PER_SEC = 60        # generation speed for the latency physics

# Round illustrative prices (§42.3). Real prices move; swap these for today's.
PRICE_INPUT_PER_M = 1.00          # $ per million input tokens
PRICE_OUTPUT_PER_M = 5.00         # $ per million output tokens

GB = 1_000 ** 3                   # we size storage in decimal GB/TB (vendor convention)
TB = 1_000 ** 4

print(f"embedding size: {EMBEDDING_BYTES:,} bytes ≈ {EMBEDDING_BYTES / 1000:.1f} KB per vector")
print(f"sanity check: 1,000,000 requests/day = {1_000_000 / SECONDS_PER_DAY:.1f} requests/s (the chapter's '≈12/s' anchor)")

## The calculator: assumptions in, a report out

Now the engine. `Assumptions` is a tiny dataclass — the handful of numbers you actually argue about in a design review. `estimate(assumptions)` returns a plain dict report. Everything downstream just reads that report, so to size *your* feature you edit one struct and re-run.

Read the arithmetic, not just the result: each line is one multiplier from the “users × requests × turns × tokens × price” chain.

In [ ]:
@dataclass(frozen=True)
class Assumptions:
    """The handful of numbers a design review actually argues about."""
    daily_active_users: int
    requests_per_user_per_day: int
    model_calls_per_request: int      # the agent loop depth
    input_tokens_per_call: int        # system prompt + history + retrieved context
    output_tokens_per_call: int
    peak_factor: float = 5.0          # peak-to-average traffic ratio


def estimate(a: Assumptions) -> dict:
    """Back-of-envelope sizing for an agentic feature. Crude on purpose."""
    # --- Traffic ---
    requests_per_day = a.daily_active_users * a.requests_per_user_per_day
    avg_requests_per_s = requests_per_day / SECONDS_PER_DAY
    peak_requests_per_s = avg_requests_per_s * a.peak_factor
    # the gateway sees every MODEL CALL, not every request — multiply by loop depth
    peak_model_calls_per_s = peak_requests_per_s * a.model_calls_per_request

    # --- Tokens (the real cost driver) ---
    input_tokens_per_request = a.model_calls_per_request * a.input_tokens_per_call
    output_tokens_per_request = a.model_calls_per_request * a.output_tokens_per_call
    input_tokens_per_day = input_tokens_per_request * requests_per_day
    output_tokens_per_day = output_tokens_per_request * requests_per_day

    # --- Dollars ---
    input_cost_per_day = input_tokens_per_day / 1_000_000 * PRICE_INPUT_PER_M
    output_cost_per_day = output_tokens_per_day / 1_000_000 * PRICE_OUTPUT_PER_M
    cost_per_day = input_cost_per_day + output_cost_per_day
    cost_per_request = cost_per_day / requests_per_day if requests_per_day else 0.0

    # --- Latency physics (sequential worst case) ---
    secs_per_call = a.output_tokens_per_call / OUTPUT_TOKENS_PER_SEC
    secs_per_request_sequential = secs_per_call * a.model_calls_per_request

    return {
        "requests_per_day": requests_per_day,
        "avg_requests_per_s": avg_requests_per_s,
        "peak_requests_per_s": peak_requests_per_s,
        "peak_model_calls_per_s": peak_model_calls_per_s,
        "input_tokens_per_day": input_tokens_per_day,
        "output_tokens_per_day": output_tokens_per_day,
        "input_cost_per_day": input_cost_per_day,
        "output_cost_per_day": output_cost_per_day,
        "cost_per_day": cost_per_day,
        "cost_per_request": cost_per_request,
        "secs_per_call": secs_per_call,
        "secs_per_request_sequential": secs_per_request_sequential,
    }


def print_report(a: Assumptions, r: dict) -> None:
    """Pretty-print a report in powers-of-ten style, not false precision."""
    print(f"{a.daily_active_users:,} DAU × {a.requests_per_user_per_day} req/user "
          f"× {a.model_calls_per_request} calls × "
          f"{a.input_tokens_per_call:,} in / {a.output_tokens_per_call:,} out tokens")
    print("-" * 64)
    print(f"  traffic : {r['requests_per_day']:,} req/day ≈ {r['avg_requests_per_s']:.1f}/s avg, "
          f"{r['peak_requests_per_s']:.0f}/s peak")
    print(f"            gateway sees ≈ {r['peak_model_calls_per_s']:.0f} model calls/s at peak")
    print(f"  tokens  : {r['input_tokens_per_day'] / 1e9:.2f} B input/day, "
          f"{r['output_tokens_per_day'] / 1e6:.0f} M output/day")
    print(f"  cost    : ${r['input_cost_per_day']:,.0f} input + ${r['output_cost_per_day']:,.0f} output "
          f"= ${r['cost_per_day']:,.0f}/day (≈ ${r['cost_per_day'] * 30:,.0f}/month)")
    print(f"            ≈ ${r['cost_per_request']:.3f} per request")
    print(f"  latency : {r['secs_per_call']:.1f} s/call × {a.model_calls_per_request} "
          f"= {r['secs_per_request_sequential']:.0f} s/request if sequential")


print("estimate() and print_report() ready.")

## 🔮 Predict: the customer-assistant scenario

Here is the chapter's exact scenario (§42.3): **10,000 daily active users × 20 requests/day**, an agent loop of **4 model calls**, each call averaging **2,500 input** and **500 output** tokens.

Before you run the next cell, **predict the \$/day**. Is it closer to \$400/day? \$4,000/day? \$40,000/day? And what's the cost *per request*? Write your guess down — then run it and compare against §42.3.

In [ ]:
scenario = Assumptions(
    daily_active_users=10_000,
    requests_per_user_per_day=20,
    model_calls_per_request=4,
    input_tokens_per_call=2_500,
    output_tokens_per_call=500,
)

report = estimate(scenario)
print_report(scenario, report)

# Confirm against the chapter's stated figures (§42.3).
print()
print("vs §42.3:")
print(f"  ≈ $4,000/day ? -> {report['cost_per_day']:,.0f}")
print(f"  ≈ $0.02/request ? -> {report['cost_per_request']:.3f}")
print(f"  ≈ 48 model calls/s at peak ? -> {report['peak_model_calls_per_s']:.0f}")

**What you just saw.** The numbers land where the chapter says: ≈ \$4,000/day (≈ \$120k/month), ≈ \$0.02 per request, ≈ 48 model calls/s at peak. Notice input and output cost are roughly equal here even though output tokens are 5× fewer — because output is priced 5× higher. Now you can *judge* the design: deflecting \$6 support tickets, \$0.02/request prints money; as a free feature on a \$10/month product, it's dead on arrival. The estimate changes the decision before any code exists.

## Latency physics: why this can't be a blocking API

The same assumptions give you a latency verdict for free. At ~60 output tokens/s, one 500-token call takes ≈ 8.3 s; four *sequential* calls take ≈ 33 s plus tool time. That number, reached before writing code, settles an architectural question (§42.3).

In [ ]:
secs = report["secs_per_request_sequential"]
print(f"one {scenario.output_tokens_per_call}-token call ≈ {report['secs_per_call']:.1f} s")
print(f"{scenario.model_calls_per_request} sequential calls ≈ {secs:.0f} s (plus tool time)")
print()
if secs > 3:
    print(f"VERDICT: {secs:.0f} s >> a few seconds — this CANNOT be a blocking request–response API.")
    print("        Stream tokens as they arrive, show intermediate steps, parallelize")
    print("        where the loop allows, and push long runs to background workers (Ch 31).")
else:
    print("VERDICT: a blocking request–response shape may be viable — but still stream first-token.")

**What you just saw.** 33 seconds is not a latency you tune away; it's a *shape* decision. First-token time is what the user feels, which is why streaming is non-negotiable for interactive agents (§25, §42.3). The arithmetic made the architecture choice for you.

## Storage: vectors are tiny, traces are not

Two storage lines, two very different conclusions. A RAG corpus of 2 million chunks with 1,536-dim float32 embeddings fits on one node; the *exhaust* (traces at ~50 KB/request) grows without bound and needs a tiering policy (§42.3, Ch 30).

In [ ]:
def storage_estimate(num_chunks: int, requests_per_day: int, trace_kb_per_request: float = 50.0) -> dict:
    """Vector-store size vs. trace growth — the two data planes' footprints."""
    vector_bytes = num_chunks * EMBEDDING_BYTES
    trace_bytes_per_day = requests_per_day * trace_kb_per_request * 1000
    trace_bytes_per_year = trace_bytes_per_day * 365
    return {
        "vector_gb": vector_bytes / GB,
        "trace_gb_per_day": trace_bytes_per_day / GB,
        "trace_tb_per_year": trace_bytes_per_year / TB,
    }


s = storage_estimate(num_chunks=2_000_000, requests_per_day=report["requests_per_day"])
print(f"vectors : 2,000,000 chunks × {EMBEDDING_BYTES / 1000:.0f} KB ≈ {s['vector_gb']:.0f} GB")
print(f"          -> comfortably one pgvector node; no tiering needed (Ch 30).")
print()
print(f"traces  : {report['requests_per_day']:,} req/day × 50 KB ≈ {s['trace_gb_per_day']:.0f} GB/day")
print(f"          -> ≈ {s['trace_tb_per_year']:.1f} TB/year: needs retention + tiering")
print(f"             (hot for 30 days, then S3).")
print()
print("vs §42.3:  ≈ 12 GB vectors, ≈ 10 GB/day, ≈ 3.6 TB/year traces.")

**What you just saw.** The two data planes have opposite shapes. The serving plane (vectors) is a fixed ≈ 12 GB — one node, done. The exhaust plane (traces) compounds to ≈ 3.6 TB/year, so it gets a retention and tiering policy on the *diagram*, not as an ops afterthought. Same corpus, same traffic, two completely different storage decisions — and the estimate is what tells them apart.

## Sensitivity sweep: which multiplier owns the bill?

The estimate's real value is *derivative*: when you change one assumption, which way does the cost move, and by how much? Re-run with loop depth 4 → 8, and with +1,000 tokens of prompt, and watch the \$/day jump. This reproduces §42.3's pitfall as a printed number.

In [ ]:
import dataclasses

base = estimate(scenario)["cost_per_day"]

# Lever 1: double the agent loop depth (4 -> 8).
deeper = dataclasses.replace(scenario, model_calls_per_request=8)
cost_deeper = estimate(deeper)["cost_per_day"]

# Lever 2: a prompt that "just" grows by 1,000 input tokens per call.
fatter = dataclasses.replace(scenario, input_tokens_per_call=scenario.input_tokens_per_call + 1_000)
cost_fatter = estimate(fatter)["cost_per_day"]

print(f"baseline                         : ${base:,.0f}/day")
print(f"loop depth 4 -> 8                : ${cost_deeper:,.0f}/day  ({cost_deeper / base:.1f}× baseline)")
print(f"prompt +1,000 input tokens/call  : ${cost_fatter:,.0f}/day  (+${cost_fatter - base:,.0f}/day)")
print()
print("§42.3's pitfall, reproduced: a prompt that 'just' grows by 1,000 tokens adds")
print(f"200,000 req/day × 4 calls × 1,000 tokens = 800M input tokens ≈ ${cost_fatter - base:,.0f}/day.")

## ⚠️ Pitfall: counting requests when the cost driver is tokens

The classic estimation error in agentic systems is reasoning about *requests* (“it's only 2.3 req/s, trivial”) when the bill is driven by *tokens*. The traffic number is genuinely modest; the token number is what costs \$120k/month. Below, the same scenario is shown both ways, side by side, so the gap is impossible to miss.

In [ ]:
r = report
print("Counted as REQUESTS (the trap):")
print(f"  {r['avg_requests_per_s']:.1f} requests/s average — 'a modest web API, nothing to worry about'")
print()
print("Counted as TOKENS (the truth):")
total_tokens_day = r["input_tokens_per_day"] + r["output_tokens_per_day"]
print(f"  {total_tokens_day / 1e9:.1f} B tokens/day — ${r['cost_per_day']:,.0f}/day ≈ ${r['cost_per_day'] * 30:,.0f}/month")
print()
print("Same system. The request view says 'trivial'; the token view says 'this is the")
print("binding constraint.' Spend = users × requests × loop turns × tokens/turn × price —")
print("four multipliers. Estimate at the token level, and re-estimate when prompts or")
print("loop behavior change.")

## 🎯 Senior lens

Precision here is worthless — chasing decimal places on an estimate is a junior tell. A senior optimizes the estimate for *speed of judgment*: reliably within 3× is enough to change a decision, and that's all a back-of-envelope is for. The point of computing \$4,000/day isn't the \$4,000; it's learning, in ten minutes, that this design is shaped by *cost*, not throughput — so the binding constraint is the model bill, and the levers to reach for are a smaller model on easy turns, fewer loop iterations, prompt caching on the static prefix, and response caching (Ch 40).

The deeper move is to read the estimate for *which multiplier dominates*. In this scenario, doubling loop depth doubles the bill while traffic stays trivial — so engineering effort belongs on loop discipline and caching, not on autoscaling the web tier. The estimate doesn't just size the system; it tells you where to spend your attention.

## Recap

- Spend scales as **users × requests × loop turns × tokens/turn × price** — estimate at the *token* level.
- Five anchors (86,400 s/day; word ≈ 1.3 tok; page ≈ 500 tok; embedding ≈ 6 KB; ≈ 60 tok/s) make it mental arithmetic.
- The chapter's scenario reproduces: ≈ **\$4,000/day**, ≈ **\$0.02/request**, ≈ 48 model calls/s peak.
- **Latency physics** alone (≈ 33 s for 4 sequential calls) rules out a blocking request–response API.
- **Storage** splits the two data planes: ≈ 12 GB vectors (one node) vs. ≈ 3.6 TB/yr traces (needs tiering).
- A **+1,000-token prompt = +\$800/day**; precision is worthless, but *within 3×* changes the decision.

## Exercises

Each exercise *changes* an assumption and asks you to predict the result before running.

1. **Halve the loop depth.** Re-run the scenario with `model_calls_per_request=2`. Predict the new \$/task and the sequential latency before you run it. Which one improves more, proportionally?
2. **Smaller-model-on-easy-turns.** Suppose 70% of turns can use a model at 1/5 the input and output price. Write a cell that blends the two prices and recomputes \$/day. How much of the bill did you reclaim?
3. **Your own feature.** Encode a feature you're actually considering as an `Assumptions` and print its report. Name its *binding multiplier* in one sentence (cost? latency? traffic? storage?).
4. **Freshness vs. storage.** Change the corpus to 10 million chunks and the trace size to 80 KB/request. Predict whether vectors or traces cross 1 TB first, then check with `storage_estimate`.

In [ ]:
# Exercise scratch space — your code here.


In [ ]:
# Exercise scratch space — your code here.


## Next

- **Next notebook:** [`42-02-run-the-method-worksheet.ipynb`](42-02-run-the-method-worksheet.ipynb) — run the full seven-step method on a system *you* pick, embedding this estimate as its Step 3.
- **Book:** revisit §42.3 (estimation) and the §42.7 worked example, whose Step 3 these numbers reproduce; §42.1 (“numbers kill bad designs”).
- **Where this leads:** the cost/latency levers live in [`blueprints/observability-stack/`](../../../blueprints/observability-stack/) (trace/eval flywheel that feeds the storage line) and the reliability seams of [`blueprints/agent-loop/`](../../../blueprints/agent-loop/); the worked support-system estimate maps directly onto the `capstone/`'s intake/resolution split.